In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!pip install -q sentence-transformers FlagEmbedding rank_bm25 pytrec_eval datasets transformers accelerate
print("✅ Packages installed")


In [ ]:
import os, json, ast, gc, warnings, pickle, random
import numpy as np
import pandas as pd
import torch
import transformers
from datasets import Dataset
from sentence_transformers import SentenceTransformer, util, CrossEncoder
from sentence_transformers.evaluation import InformationRetrievalEvaluator
from sentence_transformers.training_args import SentenceTransformerTrainingArguments
from sentence_transformers import SentenceTransformerTrainer
from sentence_transformers.losses import MultipleNegativesRankingLoss, CachedMultipleNegativesRankingLoss
from transformers import EarlyStoppingCallback
from rank_bm25 import BM25Okapi

warnings.filterwarnings('ignore')
transformers.logging.set_verbosity_error()
random.seed(42)
np.random.seed(42)

# ── Paths ────────────────────────────────────────────────────────────────────
start_path     = '/content/drive/MyDrive/TaskB/TaskB'
STAGE1_PATH    = os.path.join(start_path, 'model_stage1')
STAGE2_PATH    = os.path.join(start_path, 'model_stage2')
STAGE3_PATH    = os.path.join(start_path, 'model_stage3_hardneg')
EMB_CACHE      = os.path.join(start_path, 'corpus_emb_cache.pkl')
PRED_PATH      = os.path.join(start_path, 'predictions.trec')

# ── Model config ─────────────────────────────────────────────────────────────
BASE_MODEL     = 'BAAI/bge-base-en-v1.5'
BGE_Q_PREFIX   = "Represent this sentence for searching relevant passages: "

# ── Hyperparams ──────────────────────────────────────────────────────────────
MAX_TERMS      = 120
TRAIN_BS       = 64      # doubled from 32 — larger batch = more in-batch negatives
EVAL_BS        = 128     # doubled for faster encoding
RETRIEVAL_K    = 500     # increased from 100 — retrieve more for better map
HARD_NEG_K     = 5

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"✅ Config ready | Device: {device}")
if torch.cuda.is_available():
    print(f"   GPU : {torch.cuda.get_device_name(0)}")
    print(f"   VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")


In [ ]:
# ── Load raw data ─────────────────────────────────────────────────────────────
with open(os.path.join(start_path, 'training', 'jobid2terms.json'), 'r', encoding='utf-8') as f:
    job_dict = json.load(f)
with open(os.path.join(start_path, 'training', 'skillid2terms.json'), 'r', encoding='utf-8') as f:
    skill_dict = json.load(f)

job2skill_df = pd.read_csv(
    os.path.join(start_path, 'training', 'job2skill.tsv'),
    sep='\t', header=None, names=['job_id', 'skill_id', 'essential']
).dropna()

queries_df = pd.read_csv(os.path.join(start_path, 'validation', 'queries'), sep='\t').dropna()
corpus_df  = pd.read_csv(os.path.join(start_path, 'validation', 'corpus_elements'), sep='\t').dropna(subset=['c_id'])
qrels_df   = pd.read_csv(
    os.path.join(start_path, 'validation', 'qrels.tsv'),
    sep='\t', header=None, names=['q_id', 'dummy', 'c_id', 'relevance']
).dropna()

# ── Helper text builders ─────────────────────────────────────────────────────
def get_job_text(job_id):
    t = " ".join(job_dict.get(str(job_id), []))
    return " ".join(t.split()[:MAX_TERMS]) if t.strip() else None

def get_skill_text(skill_id):
    t = " ".join(skill_dict.get(str(skill_id), []))
    return " ".join(t.split()[:MAX_TERMS]) if t.strip() else None

# ── Validation corpus + queries ──────────────────────────────────────────────
queries_val = {
    str(row['q_id']): " ".join(str(row['jobtitle']).split()[:MAX_TERMS])
    if pd.notna(row['jobtitle']) else "unknown"
    for _, row in queries_df.iterrows()
}

corpus_val = {}
for _, row in corpus_df.iterrows():
    cid = str(row['c_id'])
    try:
        aliases = ast.literal_eval(row['skill_aliases'])
        text = " ".join(" ".join(aliases).split()[:MAX_TERMS])
    except:
        text = " ".join(str(row['skill_aliases']).split()[:MAX_TERMS])
    corpus_val[cid] = text if text.strip() else "unknown"

queries_eval  = {}
relevant_docs = {}
for qid, qtext in queries_val.items():
    pos = qrels_df[(qrels_df['q_id'].astype(str)==qid)&(qrels_df['relevance']>0)]['c_id'].astype(str).tolist()
    if pos:
        queries_eval[qid]  = qtext
        relevant_docs[qid] = set(pos)

# ── Build job→skills map (all skills per job, with essential flag) ────────────
job2all_skills = {}   # {job_id: [(skill_id, essential_bool), ...]}
for _, row in job2skill_df.iterrows():
    jid = str(row['job_id'])
    sid = str(row['skill_id'])
    ess = str(row['essential']).strip().lower() == 'essential'
    job2all_skills.setdefault(jid, []).append((sid, ess))

print(f"✅ Data loaded")
print(f"   Jobs in training  : {len(job2all_skills)}")
print(f"   Val queries       : {len(queries_eval)}")
print(f"   Corpus skills     : {len(corpus_val)}")


In [ ]:
# ── Stage 1 dataset: all job→skill pairs ─────────────────────────────────────
anchors_all, positives_all = [], []

for jid, skill_list in job2all_skills.items():
    job_text = get_job_text(jid)
    if not job_text:
        continue
    for sid, ess in skill_list:
        skill_text = get_skill_text(sid)
        if skill_text:
            anchors_all.append(job_text)
            positives_all.append(skill_text)
from transformers.trainer_utils import get_last_checkpoint


train_ds_all = Dataset.from_dict({"anchor": anchors_all, "positive": positives_all})
print(f"✅ Stage 1 dataset: {len(train_ds_all)} pairs")


In [ ]:
# ── Stage 2 dataset: essential pairs only (curriculum sharpening) ─────────────
anchors_ess, positives_ess = [], []

for jid, skill_list in job2all_skills.items():
    job_text = get_job_text(jid)
    if not job_text:
        continue
    for sid, ess in skill_list:
        if not ess:
            continue
        skill_text = get_skill_text(sid)
        if skill_text:
            anchors_ess.append(job_text)
            positives_ess.append(skill_text)

train_ds_ess = Dataset.from_dict({"anchor": anchors_ess, "positive": positives_ess})
print(f"✅ Stage 2 dataset: {len(train_ds_ess)} essential pairs")


In [ ]:
# ── STAGE 1: Train on ALL pairs ──────────────────────────────────────────────
evaluator  = InformationRetrievalEvaluator(
    queries_eval, corpus_val, relevant_docs,
    name='val_ir',
    mrr_at_k=[10, 100],
    ndcg_at_k=[10, 100],
    map_at_k=[100, 500],   # track map@500 to match full-ranking behaviour
    precision_recall_at_k=[5, 10, 100],
)
model      = SentenceTransformer(BASE_MODEL)
loss_fn    = CachedMultipleNegativesRankingLoss(model, mini_batch_size=32)  # bigger effective batch
eval_steps = max(1, len(train_ds_all) // (TRAIN_BS * 10))

args1 = SentenceTransformerTrainingArguments(
    output_dir=STAGE1_PATH,
    num_train_epochs=3,
    per_device_train_batch_size=TRAIN_BS,
    eval_strategy="steps", eval_steps=eval_steps,
    save_strategy="steps", save_steps=eval_steps,
    save_total_limit=3,
    load_best_model_at_end=True,
    metric_for_best_model="eval_val_ir_cosine_map@500",
    warmup_ratio=0.1,
    fp16=True,
    dataloader_num_workers=2,
    logging_steps=eval_steps,
)

trainer1 = SentenceTransformerTrainer(
    model=model, args=args1,
    train_dataset=train_ds_all,
    evaluator=evaluator, loss=loss_fn,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=5)]
)

print("🚀 Stage 1: Training on ALL pairs...")
trainer1.train(resume_from_checkpoint=None)
model.save_pretrained(STAGE1_PATH)
print(f"✅ Stage 1 done → {STAGE1_PATH}")


In [ ]:
# ── STAGE 2: Fine-tune on ESSENTIAL pairs ────────────────────────────────────
from sentence_transformers.evaluation import InformationRetrievalEvaluator
import glob

evaluator = InformationRetrievalEvaluator(
    queries_eval, corpus_val, relevant_docs,
    name='val_ir',
    mrr_at_k=[10, 100],
    ndcg_at_k=[10, 100],
    map_at_k=[100, 500],
    precision_recall_at_k=[5, 10, 100],
)

_ckpts = sorted(glob.glob(f"{STAGE1_PATH}/checkpoint-*"), key=lambda p: int(p.split("-")[-1]))
model = SentenceTransformer(STAGE1_PATH if os.path.exists(f"{STAGE1_PATH}/config.json") else _ckpts[-1])

loss_fn    = CachedMultipleNegativesRankingLoss(model, mini_batch_size=32)
eval_steps = max(1, len(train_ds_ess) // (TRAIN_BS * 10))

args2 = SentenceTransformerTrainingArguments(
    output_dir=STAGE2_PATH,
    num_train_epochs=3,
    per_device_train_batch_size=TRAIN_BS,
    eval_strategy="steps", eval_steps=eval_steps,
    save_strategy="steps", save_steps=eval_steps,
    save_total_limit=3,
    load_best_model_at_end=True,
    metric_for_best_model="eval_val_ir_cosine_map@500",
    warmup_ratio=0.05,
    learning_rate=1e-5,
    fp16=True,
    dataloader_num_workers=2,
    logging_steps=eval_steps,
)

trainer2 = SentenceTransformerTrainer(
    model=model, args=args2,
    train_dataset=train_ds_ess,
    evaluator=evaluator, loss=loss_fn,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=5)]
)

print("🚀 Stage 2: Training on ESSENTIAL pairs only...")
trainer2.train(resume_from_checkpoint=None)
model.save_pretrained(STAGE2_PATH)
print(f"✅ Stage 2 done → {STAGE2_PATH}")


In [ ]:
# ── HARD NEGATIVE MINING ──────────────────────────────────────────────────────
# Use the stage-2 model to find skills that LOOK like correct answers but AREN'T
# These are far more informative training signals than random negatives
#
# Method:
#   For each job → encode → retrieve top-50 skills → remove true positives
#                         → keep top-HARD_NEG_K wrong skills as hard negatives
#
# Memory strategy: encode all corpus first, then query in batches

print("Loading Stage 2 model for mining...")
miner_model = SentenceTransformer(STAGE1_PATH)
miner_model.eval()

# ── Encode ALL skills in corpus ──────────────────────────────────────────────
all_skill_ids   = list(skill_dict.keys())
all_skill_texts = [get_skill_text(sid) for sid in all_skill_ids]
valid_mask      = [t is not None for t in all_skill_texts]
all_skill_ids   = [s for s, v in zip(all_skill_ids, valid_mask) if v]
all_skill_texts = [t for t, v in zip(all_skill_texts, valid_mask) if v]

print(f"Encoding {len(all_skill_texts)} skills for mining...")
skill_embeddings = miner_model.encode(
    all_skill_texts, batch_size=EVAL_BS,
    convert_to_tensor=True, normalize_embeddings=True,
    show_progress_bar=True
)

# ── Mine hard negatives ───────────────────────────────────────────────────────
anchors_hn, positives_hn, negatives_hn = [], [], []
unique_jobs = list(job2all_skills.keys())

print(f"Mining hard negatives for {len(unique_jobs)} jobs...")
for i, jid in enumerate(unique_jobs):
    if (i+1) % 500 == 0:
        print(f"  {i+1}/{len(unique_jobs)} jobs processed...")

    job_text = get_job_text(jid)
    if not job_text:
        continue

    true_sids = set(sid for sid, _ in job2all_skills[jid])

    # encode query
    q_emb = miner_model.encode(
        BGE_Q_PREFIX + job_text,
        convert_to_tensor=True, normalize_embeddings=True, show_progress_bar=False
    )
    sims = util.dot_score(q_emb, skill_embeddings)[0].cpu().numpy()

    # rank all skills, exclude true positives
    ranked_idxs = np.argsort(sims)[::-1]
    hard_neg_idxs = [idx for idx in ranked_idxs if all_skill_ids[idx] not in true_sids][:HARD_NEG_K]

    if not hard_neg_idxs:
        continue

    # for each true positive, pair with each hard negative
    for sid, ess in job2all_skills[jid]:
        pos_text = get_skill_text(sid)
        if not pos_text:
            continue
        for neg_idx in hard_neg_idxs:
            anchors_hn.append(job_text)
            positives_hn.append(pos_text)
            negatives_hn.append(all_skill_texts[neg_idx])

print(f"✅ Mined {len(anchors_hn)} hard negative triplets")

# free miner model memory
del miner_model, skill_embeddings
gc.collect()
torch.cuda.empty_cache()

# save to disk so you don't lose it on crash
hn_cache = os.path.join(start_path, 'hard_negatives_cache.pkl')
with open(hn_cache, 'wb') as f:
    pickle.dump({'anchors': anchors_hn, 'positives': positives_hn, 'negatives': negatives_hn}, f)
print(f"💾 Hard negatives cached → {hn_cache}")


In [ ]:
# ── (RECOVERY) Load hard negatives from cache if you crashed ─────────────────
# Skip this cell if you just ran the mining cell above
# Uncomment below if you need to reload:

hn_cache = os.path.join(start_path, 'hard_negatives_cache.pkl')
with open(hn_cache, 'rb') as f:
    hn_data = pickle.load(f)
anchors_hn   = hn_data['anchors']
positives_hn = hn_data['positives']
negatives_hn = hn_data['negatives']
print(f"✅ Loaded {len(anchors_hn)} hard negative triplets from cache")
print("(Skipped — cache recovery not needed)")


In [ ]:
# ── STAGE 3: SKIPPED ─────────────────────────────────────────────────────────
# Hard negative training consistently degraded map and mrr on this dataset.
# Stage 2 model is the best checkpoint. Inference runs directly from STAGE2_PATH.
print("⏭️  Stage 3 skipped — using Stage 2 model for inference")


In [ ]:
# ── Encode corpus + build BM25 index ─────────────────────────────────────────
best_model = SentenceTransformer(STAGE1_PATH)   # Stage 2 is best
best_model.eval()

cids         = list(corpus_val.keys())
corpus_texts = [corpus_val[cid] for cid in cids]

print(f"Encoding {len(corpus_texts)} corpus skills...")
corpus_embeddings = best_model.encode(
    corpus_texts, batch_size=EVAL_BS,
    convert_to_tensor=True, normalize_embeddings=True,
    show_progress_bar=True
)

print("Building BM25 index...")
tokenized_corpus = [t.lower().split() for t in corpus_texts]
bm25 = BM25Okapi(tokenized_corpus)

with open(EMB_CACHE, 'wb') as f:
    pickle.dump({'cids': cids, 'embeddings': corpus_embeddings.cpu().numpy()}, f)
print(f"✅ Corpus encoded | BM25 ready | Cache → {EMB_CACHE}")


In [ ]:
# ── (RECOVERY) Reload corpus embeddings from cache if crashed ────────────────
# Run this cell instead of the encoding cell above if you crashed

# with open(EMB_CACHE, 'rb') as f:
#     cache = pickle.load(f)
# cids             = cache['cids']
# corpus_texts     = [corpus_val[cid] for cid in cids]
# corpus_embeddings= torch.tensor(cache['embeddings']).to(device)
# tokenized_corpus = [t.lower().split() for t in corpus_texts]
# bm25             = BM25Okapi(tokenized_corpus)
# best_model       = SentenceTransformer(STAGE3_PATH)
# print("✅ Reloaded from cache")
print("(Skipped — cache recovery not needed)")


In [ ]:
# ── FULL INFERENCE PIPELINE ──────────────────────────────────────────────────
# Dense BGE + BM25 → RRF → top-500
# No PRF, no reranker — both hurt domain-specific retrieval

def hybrid_retrieve(query_text, top_k=RETRIEVAL_K):
    q_emb = best_model.encode(
        BGE_Q_PREFIX + query_text,
        convert_to_tensor=True, normalize_embeddings=True, show_progress_bar=False
    )
    dense_scores = util.dot_score(q_emb, corpus_embeddings)[0].cpu().numpy()
    dense_ranked = np.argsort(dense_scores)[::-1][:top_k*2]

    bm25_scores  = bm25.get_scores(query_text.lower().split())
    bm25_ranked  = np.argsort(bm25_scores)[::-1][:top_k*2]

    K = 60
    rrf = {}
    for rank, idx in enumerate(dense_ranked):
        rrf[idx] = rrf.get(idx, 0) + 1.0/(K + rank + 1)
    for rank, idx in enumerate(bm25_ranked):
        rrf[idx] = rrf.get(idx, 0) + 1.0/(K + rank + 1)

    top_idxs = sorted(rrf, key=rrf.get, reverse=True)[:top_k]
    return [(cids[i], rrf[i]) for i in top_idxs]

print("🚀 Running inference (Hybrid Dense+BM25, top-500)...")
results = []

for i, (qid, query_text) in enumerate(queries_val.items()):
    if (i+1) % 100 == 0:
        print(f"  {i+1}/{len(queries_val)} queries done...")

    final_ranking = hybrid_retrieve(query_text)

    for rank, (cid, score) in enumerate(final_ranking):
        results.append(f"{qid}\tQ0\t{cid}\t{rank+1}\t{float(score)}\tbge_hybrid_500")

with open(PRED_PATH, 'w') as f:
    f.write("\n".join(results))

print(f"✅ Written {len(results)} lines → {PRED_PATH}")


In [ ]:
pip install pytrec_eval

In [ ]:
import os
import pytrec_eval
import pandas as pd
# ── Encode corpus + build BM25 index ─────────────────────────────────────────


start_path = '/content/drive/MyDrive/TaskB/TaskB'
qrels_path = os.path.join(start_path, 'validation', 'qrels.tsv')
trec_path = os.path.join(start_path, 'predictions.trec')

qrels_df = pd.read_csv(qrels_path, sep='\t', header=None, names=['q_id', 'dummy', 'c_id', 'relevance'])
qrels = {}
for _, row in qrels_df.iterrows():
    qid = str(row['q_id'])
    if qid not in qrels:
        qrels[qid] = {}
    qrels[qid][str(row['c_id'])] = int(row['relevance'])

run = {}
with open(trec_path, 'r') as f:
    for line in f:
        qid, _, docid, rank, score, _ = line.strip().split()
        if qid not in run:
            run[qid] = {}
        run[qid][docid] = float(score)

evaluator = pytrec_eval.RelevanceEvaluator(qrels, {'ndcg_cut_10'})
results = evaluator.evaluate(run)

ndcg_scores = [query_measures['ndcg_cut_10'] for query_measures in results.values()]
average_ndcg = sum(ndcg_scores) / len(ndcg_scores)
print(average_ndcg)

In [ ]:
!git clone https://github.com/TalentCLEF/talentclef26_evaluation_script.git
!pip install -r /content/talentclef26_evaluation_script/requirements.txt

In [ ]:
import os
import subprocess

start_path = '/content/drive/MyDrive/TaskB/TaskB'
old_trec_path = os.path.join(start_path, 'predictions.trec')
new_trec_path = os.path.join(start_path, 'predictions_formatted.trec')

with open(old_trec_path, 'r') as infile, open(new_trec_path, 'w') as outfile:
    for line in infile:
        outfile.write(line.replace('\t', ' '))

qrels_file = os.path.join(start_path, 'validation', 'qrels.tsv')

command = [
    "python",
    "/content/talentclef26_evaluation_script/talentclef_evaluate.py",
    "--task", "B",
    "--qrels", qrels_file,
    "--run", new_trec_path
]

result = subprocess.run(command, capture_output=True, text=True)
print(result.stdout)

In [ ]:
!zip -j /content/drive/MyDrive/TaskB/TaskB/submission.zip /content/drive/MyDrive/TaskB/TaskB/predictions_formatted.trec

In [ ]:
from google.colab import files

files.download('/content/drive/MyDrive/TaskB/TaskB/submission.zip')

In [ ]:
import shutil
import os

start_path = '/content/drive/MyDrive/TaskB/TaskB'
# This is the formatted file you generated a few steps ago
old_trec_path = os.path.join(start_path, 'predictions_formatted.trec')
# New file following the run_xx-xx_teamname.trec rule
new_trec_path = os.path.join(start_path, 'run_en-en_Olive.trec')

shutil.copy(old_trec_path, new_trec_path)

In [ ]:
!zip -j /content/drive/MyDrive/TaskB/TaskB/submission_Olive.zip /content/drive/MyDrive/TaskB/TaskB/run_en-en_Olive.trec

In [ ]:
from google.colab import files

files.download('/content/drive/MyDrive/TaskB/TaskB/submission_Olive.zip')

In [ ]:
from google.colab import files

files.download('/content/drive/MyDrive/TaskB/TaskB/submission_test_Olive.zip')